In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
)
import warnings
warnings.filterwarnings('ignore')

In [5]:
# ============================================================
# 配置路径
# ============================================================
STAGE1_DIR = "/content/drive/MyDrive/s1/stage1_artifacts_full_64_head_T_C_gated_chronological"

# ============================================================
# 加载三个 split 的 npz 文件
# ============================================================
print("=" * 60)
print("加载 Stage 1 嵌入...")
print("=" * 60)

data_splits = {}
for split in ["train", "val", "test"]:
    path = f"{STAGE1_DIR}/stage1_{split}_embeddings.npz"
    data = np.load(path, allow_pickle=True)

    z = data["z"].astype(np.float32)           # 嵌入矩阵
    labels = data["label"].astype(np.int64)     # 标签
    flow_ids = data["flow_id"].astype(np.int64) # 流ID

    data_splits[split] = {
        "z": z,
        "labels": labels,
        "flow_ids": flow_ids,
    }

    n_pos = int(labels.sum())
    n_neg = int(len(labels) - n_pos)
    ratio = n_neg / max(n_pos, 1)
    print(f"{split:>5}: {len(labels):>6} 样本 | 正样本: {n_pos:>4} | 负样本: {n_neg:>5} | 不平衡比: {ratio:.1f}:1 | 嵌入维度: {z.shape[1]}")

# 提取各 split
z_train = data_splits["train"]["z"]
z_train_scaled = z_train
y_train = data_splits["train"]["labels"]
z_val = data_splits["val"]["z"]
z_val_scaled = z_val
y_val = data_splits["val"]["labels"]
z_test = data_splits["test"]["z"]
z_test_scaled = z_test
y_test = data_splits["test"]["labels"]

加载 Stage 1 嵌入...
train:  27156 样本 | 正样本:  899 | 负样本: 26257 | 不平衡比: 29.2:1 | 嵌入维度: 128
  val:   3847 样本 | 正样本:  132 | 负样本:  3715 | 不平衡比: 28.1:1 | 嵌入维度: 128
 test:   7668 样本 | 正样本:  260 | 负样本:  7408 | 不平衡比: 28.5:1 | 嵌入维度: 128


In [6]:
# ============================================================
# 通用评估函数
# ============================================================
def evaluate_model(name, y_true, y_pred, y_prob):
    """统一格式输出评估结果"""
    f1_c1 = f1_score(y_true, y_pred, pos_label=1)
    rec_c1 = recall_score(y_true, y_pred, pos_label=1)
    prec_c1 = precision_score(y_true, y_pred, pos_label=1)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    auc = roc_auc_score(y_true, y_prob)

    print(f"  [{name}]")
    print(f"    Macro F1:        {f1_macro:.4f}")
    print(f"    F1  (Class 1):   {f1_c1:.4f}")
    print(f"    Recall (Class 1):   {rec_c1:.4f}")
    print(f"    Precision (Class 1):{prec_c1:.4f}")
    print(f"    AUC:              {auc:.4f}")
    return f1_c1, rec_c1, prec_c1, f1_macro, auc

# ============================================================
# 基线 1: Logistic Regression
# ============================================================
print("\n" + "=" * 60)
print("基线 1: Logistic Regression（线性探测）")
print("=" * 60)

results_lr = {}
for cw_name, cw in [("balanced", "balanced"), ("no_weight", None)]:
    print(f"\n--- class_weight = {cw_name} ---")
    clf = LogisticRegression(
        class_weight=cw,
        max_iter=5000,
        solver='lbfgs',
        random_state=42,
    )
    clf.fit(z_train_scaled, y_train)

    # 验证集
    y_val_pred = clf.predict(z_val_scaled)
    y_val_prob = clf.predict_proba(z_val_scaled)[:, 1]
    results_lr[f"val_{cw_name}"] = evaluate_model("验证集", y_val, y_val_pred, y_val_prob)

    # 测试集
    y_test_pred = clf.predict(z_test_scaled)
    y_test_prob = clf.predict_proba(z_test_scaled)[:, 1]
    results_lr[f"test_{cw_name}"] = evaluate_model("测试集", y_test, y_test_pred, y_test_prob)

# ============================================================
# 基线 2: Logistic Regression + 阈值搜索
# ============================================================
print("\n" + "=" * 60)
print("基线 2: Logistic Regression + 阈值优化（优化 F1-Class1）")
print("=" * 60)

clf_lr = LogisticRegression(class_weight="balanced", max_iter=5000, solver='lbfgs', random_state=42)
clf_lr.fit(z_train_scaled, y_train)

# 在验证集上搜索最佳阈值
y_val_prob_lr = clf_lr.predict_proba(z_val_scaled)[:, 1]
best_thresh_lr = 0.5
best_val_f1_c1 = 0

for t in np.linspace(0.01, 0.99, 99):
    pred_t = (y_val_prob_lr >= t).astype(int)
    f1_c1 = f1_score(y_val, pred_t, pos_label=1)
    if f1_c1 > best_val_f1_c1:
        best_val_f1_c1 = f1_c1
        best_thresh_lr = t

print(f"[验证集] 最佳阈值: {best_thresh_lr:.4f}, 最佳 F1-Class1: {best_val_f1_c1:.4f}")

# 用最佳阈值在测试集上评估
y_test_prob_lr = clf_lr.predict_proba(z_test_scaled)[:, 1]
y_test_pred_thresh = (y_test_prob_lr >= best_thresh_lr).astype(int)
results_lr_thresh = evaluate_model("测试集（阈值优化）", y_test, y_test_pred_thresh, y_test_prob_lr)

# ============================================================
# 基线 3: Random Forest
# ============================================================
print("\n" + "=" * 60)
print("基线 3: Random Forest（非线性探测）")
print("=" * 60)

results_rf = {}

for cw_name, cw in [("balanced", "balanced"), ("balanced_subsample", "balanced_subsample"), ("no_weight", None)]:
    print(f"\n--- class_weight = {cw_name} ---")
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        class_weight=cw,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(z_train, y_train)  # RF 不需要标准化

    # 验证集
    y_val_pred = rf.predict(z_val)
    y_val_prob = rf.predict_proba(z_val)[:, 1]
    results_rf[f"val_{cw_name}"] = evaluate_model("验证集", y_val, y_val_pred, y_val_prob)

    # 测试集
    y_test_pred = rf.predict(z_test)
    y_test_prob = rf.predict_proba(z_test)[:, 1]
    results_rf[f"test_{cw_name}"] = evaluate_model("测试集", y_test, y_test_pred, y_test_prob)

# ============================================================
# 基线 4: Random Forest + 阈值搜索
# ============================================================
print("\n" + "=" * 60)
print("基线 4: Random Forest + 阈值优化（优化 F1-Class1）")
print("=" * 60)

rf_best = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_best.fit(z_train, y_train)

y_val_prob_rf = rf_best.predict_proba(z_val)[:, 1]
best_thresh_rf = 0.5
best_val_f1_c1_rf = 0

for t in np.linspace(0.01, 0.99, 99):
    pred_t = (y_val_prob_rf >= t).astype(int)
    f1_c1 = f1_score(y_val, pred_t, pos_label=1)
    if f1_c1 > best_val_f1_c1_rf:
        best_val_f1_c1_rf = f1_c1
        best_thresh_rf = t

print(f"[验证集] 最佳阈值: {best_thresh_rf:.4f}, 最佳 F1-Class1: {best_val_f1_c1_rf:.4f}")

y_test_prob_rf = rf_best.predict_proba(z_test)[:, 1]
y_test_pred_rf_thresh = (y_test_prob_rf >= best_thresh_rf).astype(int)
results_rf_thresh = evaluate_model("测试集（阈值优化）", y_test, y_test_pred_rf_thresh, y_test_prob_rf)


基线 1: Logistic Regression（线性探测）

--- class_weight = balanced ---
  [验证集]
    Macro F1:        0.8279
    F1  (Class 1):   0.6707
    Recall (Class 1):   0.8409
    Precision (Class 1):0.5578
    AUC:              0.9671
  [测试集]
    Macro F1:        0.7654
    F1  (Class 1):   0.5510
    Recall (Class 1):   0.6962
    Precision (Class 1):0.4559
    AUC:              0.9439

--- class_weight = no_weight ---
  [验证集]
    Macro F1:        0.8356
    F1  (Class 1):   0.6810
    Recall (Class 1):   0.5985
    Precision (Class 1):0.7900
    AUC:              0.9695
  [测试集]
    Macro F1:        0.8143
    F1  (Class 1):   0.6391
    Recall (Class 1):   0.5346
    Precision (Class 1):0.7943
    AUC:              0.9469

基线 2: Logistic Regression + 阈值优化（优化 F1-Class1）
[验证集] 最佳阈值: 0.9000, 最佳 F1-Class1: 0.7302
  [测试集（阈值优化）]
    Macro F1:        0.8207
    F1  (Class 1):   0.6525
    Recall (Class 1):   0.5885
    Precision (Class 1):0.7321
    AUC:              0.9439

基线 3: Random Forest（非线性探测）

-

In [7]:
# ============================================================
# 对比：Stage 2 结果
# ============================================================
print("\n" + "=" * 60)
print("对比：Stage 2 结果（来自你之前的运行日志）")
print("=" * 60)

stage2_test = {
    "F1 (Class 1)": 0.6819,
    "Recall (Class 1)": 0.6308,
    "Precision (Class 1)": 0.7421,
    "AUC": 0.9539,
    "Macro F1": 0.8358,
}

stage2_val_best = {
    "F1 (Class 1)": 0.6268,   # Epoch 20
    "Recall (Class 1)": 0.6818,
}

print(f"  [Stage 2 - 验证集最优 (Epoch 20)]")
print(f"    F1 (Class 1):        {stage2_val_best['F1 (Class 1)']:.4f}")
print(f"    Recall (Class 1):    {stage2_val_best['Recall (Class 1)']:.4f}")
print(f"\n  [Stage 2 - 测试集]")
print(f"    Macro F1:             {stage2_test['Macro F1']:.4f}")
print(f"    F1 (Class 1):         {stage2_test['F1 (Class 1)']:.4f}")
print(f"    Recall (Class 1):     {stage2_test['Recall (Class 1)']:.4f}")
print(f"    Precision (Class 1):  {stage2_test['Precision (Class 1)']:.4f}")
print(f"    AUC:                  {stage2_test['AUC']:.4f}")

# ============================================================
# 汇总对比表
# ============================================================
print("\n" + "=" * 60)
print("汇总对比：Stage 1 嵌入质量 vs. Stage 2 完整模型")
print("=" * 60)

print(f"\n{'方法':<45} {'Test F1-C1':>11} {'Test Recall-C1':>14} {'Test AUC':>10}")
print("-" * 80)

# Stage 1 + LR (balanced, default threshold)
f1_c1_lr = f1_score(y_test, clf_lr.predict(z_test_scaled), pos_label=1)
rec_c1_lr = recall_score(y_test, clf_lr.predict(z_test_scaled), pos_label=1)
auc_lr = roc_auc_score(y_test, y_test_prob_lr)
print(f"{'Stage1 + LR (balanced, thresh=0.5)':<45} {f1_c1_lr:>11.4f} {rec_c1_lr:>14.4f} {auc_lr:>10.4f}")

# Stage 1 + LR (balanced, optimized threshold)
print(f"{'Stage1 + LR (balanced, optimized thresh)':<45} {results_lr_thresh[0]:>11.4f} {results_lr_thresh[1]:>14.4f} {results_lr_thresh[4]:>10.4f}")

# Stage 1 + RF (best)
rf_final = RandomForestClassifier(n_estimators=200, max_depth=15, class_weight="balanced", random_state=42, n_jobs=-1)
rf_final.fit(z_train, y_train)
y_test_pred_rf = rf_final.predict(z_test)
y_test_prob_rf_final = rf_final.predict_proba(z_test)[:, 1]
f1_c1_rf = f1_score(y_test, y_test_pred_rf, pos_label=1)
rec_c1_rf = recall_score(y_test, y_test_pred_rf, pos_label=1)
auc_rf = roc_auc_score(y_test, y_test_prob_rf_final)
print(f"{'Stage1 + RF (balanced, thresh=0.5)':<45} {f1_c1_rf:>11.4f} {rec_c1_rf:>14.4f} {auc_rf:>10.4f}")

# Stage 1 + RF (best, optimized threshold)
print(f"{'Stage1 + RF (balanced, optimized thresh)':<45} {results_rf_thresh[0]:>11.4f} {results_rf_thresh[1]:>14.4f} {results_rf_thresh[4]:>10.4f}")

# Stage 2
print(f"{'Stage 2 Transformer (source_host)':<45} {stage2_test['F1 (Class 1)']:>11.4f} {stage2_test['Recall (Class 1)']:>14.4f} {stage2_test['AUC']:>10.4f}")

# ============================================================
# 诊断总结
# ============================================================
print("\n" + "=" * 60)
print("诊断总结与建议")
print("=" * 60)

# 自动判断
gap_lr_vs_stage2 = stage2_test["Recall (Class 1)"] - rec_c1_lr
gap_rf_vs_stage2 = stage2_test["Recall (Class 1)"] - rec_c1_rf

print(f"""
关键发现：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1️⃣  Stage1 嵌入的线性可分性：
   LR balanced (thresh=0.5) → Recall: {rec_c1_lr:.4f}, F1: {f1_c1_lr:.4f}
   LR balanced (阈值优化)     → Recall: {results_lr_thresh[1]:.4f}, F1: {results_lr_thresh[0]:.4f}

2️⃣  Stage1 嵌入的非线性判别力：
   RF balanced (thresh=0.5) → Recall: {rec_c1_rf:.4f}, F1: {f1_c1_rf:.4f}
   RF balanced (阈值优化)     → Recall: {results_rf_thresh[1]:.4f}, F1: {results_rf_thresh[0]:.4f}

3️⃣  Stage2 Transformer 增益：
   Recall gap (Stage2 - LR):  {gap_lr_vs_stage2:+.4f}
   Recall gap (Stage2 - RF):  {gap_rf_vs_stage2:+.4f}
""")

if rec_c1_lr > 0.60:
    print("✅ Stage 1 嵌入本身具有较好的线性判别力（Recall > 0.60）")
else:
    print("⚠️  Stage 1 嵌入的线性判别力不足（Recall < 0.60），需要加强 Stage 1 训练")

if results_rf_thresh[1] > stage2_test["Recall (Class 1)"]:
    print("⚠️  Stage1 + RF 优于 Stage2 Transformer！Stage 2 的上下文建模未带来正向增益")
    print("   → 建议：(1) 加强 Stage2 正则化 (2) 检查是否存在标签泄漏 (3) 降低模型复杂度")
elif results_rf_thresh[1] > stage2_test["Recall (Class 1)"] - 0.03:
    print("⚡ Stage1 + RF 与 Stage2 表现接近，Stage 2 上下文建模增益有限")
    print("   → 建议：优化 Stage1 的包级时序建模，或增强 Stage2 的跨流特征交互")
else:
    gap = stage2_test["Recall (Class 1)"] - results_rf_thresh[1]
    print(f"✅ Stage2 Transformer 比 Stage1 + RF 提升 Recall 达 {gap:.4f}，上下文建模有效")
    print("   → 建议：继续优化 Stage2 的阈值策略和正则化以进一步提升")

if results_lr_thresh[1] > rec_c1_lr + 0.05:
    print(f"\n💡 阈值优化为 LR 带来了 {results_lr_thresh[1]-rec_c1_lr:.4f} 的 Recall 提升")
    print("   → Stage 2 也应采用阈值优化策略，当前 Stage2 阈值 0.94 过于保守")

print("\n" + "=" * 60)


对比：Stage 2 结果（来自你之前的运行日志）
  [Stage 2 - 验证集最优 (Epoch 20)]
    F1 (Class 1):        0.6268
    Recall (Class 1):    0.6818

  [Stage 2 - 测试集]
    Macro F1:             0.8358
    F1 (Class 1):         0.6819
    Recall (Class 1):     0.6308
    Precision (Class 1):  0.7421
    AUC:                  0.9539

汇总对比：Stage 1 嵌入质量 vs. Stage 2 完整模型

方法                                             Test F1-C1 Test Recall-C1   Test AUC
--------------------------------------------------------------------------------
Stage1 + LR (balanced, thresh=0.5)                 0.5510         0.6962     0.9439
Stage1 + LR (balanced, optimized thresh)           0.6525         0.5885     0.9439
Stage1 + RF (balanced, thresh=0.5)                 0.6036         0.6385     0.9526
Stage1 + RF (balanced, optimized thresh)           0.5975         0.6423     0.9526
Stage 2 Transformer (source_host)                  0.6819         0.6308     0.9539

诊断总结与建议

关键发现：
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
# 直接比较 Stage1 嵌入的分类能力 vs Stage2 输出
import numpy as np
from sklearn.metrics import roc_auc_score

STAGE1_DIR = "/content/drive/MyDrive/s1/stage1_artifacts_full_64_head_T_C_gated_chronological"

# 加载 Stage1 测试集嵌入
data = np.load(f"{STAGE1_DIR}/stage1_test_embeddings.npz", allow_pickle=True)
z_test = data["z"]
y_test = data["label"]

# 用最简单的均值向量作为"攻击模板"
attack_template = z_train[y_train == 1].mean(axis=0)  # 所有训练攻击样本的均值
normal_template = z_train[y_train == 0].mean(axis=0)  # 所有训练正常样本的均值

# 余弦相似度分类（无需训练！）
from numpy.linalg import norm
cos_attack = np.dot(z_test, attack_template) / (norm(z_test, axis=1) * norm(attack_template))
cos_normal = np.dot(z_test, normal_template) / (norm(z_test, axis=1) * norm(normal_template))

# 简单决策：更接近攻击模板 → 预测为攻击
y_score_simple = cos_attack - cos_normal  # 正值 = 更像攻击
auc_simple = roc_auc_score(y_test, y_score_simple)

print(f"\n{'='*60}")
print(f"Stage1 嵌入 + 余弦相似度（零训练）: AUC = {auc_simple:.4f}")
print(f"Stage2 Transformer（完整训练）:     AUC = 0.9539")
print(f"{'='*60}")

if auc_simple > 0.90:
    print("\n✅ Stage1 嵌入质量优秀——连无需训练的余弦相似度都能达到高 AUC")
    print("❌ 问题确定在 Stage2 的上下文建模")


Stage1 嵌入 + 余弦相似度（零训练）: AUC = 0.9656
Stage2 Transformer（完整训练）:     AUC = 0.9539

✅ Stage1 嵌入质量优秀——连无需训练的余弦相似度都能达到高 AUC
❌ 问题确定在 Stage2 的上下文建模
